# Acoustic Co-Design Demo

Joint optimization of a thin-plate resonator's thickness field, piezo actuator placement, and modal LQR gains to match a target acoustic frequency response.

This notebook runs three things end to end:
1. **Sanity check**: compare numerical eigenfrequencies at uniform thickness against the analytical simply-supported formula.
2. **Passive baseline**: optimize only the thickness field with feedback gains forced to zero.
3. **Full co-design**: optimize thickness, actuator positions, and LQR weights jointly.

In [ ]:
# If running on Colab, uncomment to install dependencies and clone:
# !pip install -q jax optax matplotlib
# !git clone https://github.com/chrisjinyu/acoustic-codesign.git
# %cd acoustic-codesign

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

import codesign
import plots
from codesign import cfg

print(f'JAX backend: {jax.default_backend()}')
print(f'Plate: {cfg.Lx*1e3:.0f} x {cfg.Ly*1e3:.0f} mm, grid {cfg.Nx} x {cfg.Ny}')
print(f'Design vars: {cfg.M**2} thickness coeffs + {cfg.n_actuators*2} positions + 2 LQR weights')

## 1. Sanity check: uniform plate eigenfrequencies

For a simply-supported rectangular plate at uniform thickness $h_0$, the analytical natural frequencies are

$$\omega_{pq} = \sqrt{\frac{D_0}{\rho h_0}} \left[ \left(\frac{p\pi}{L_x}\right)^2 + \left(\frac{q\pi}{L_y}\right)^2 \right]$$

with $D_0 = E h_0^3 / [12(1-\nu^2)]$. The first few numerical eigenvalues should agree to within a few percent.

In [ ]:
c0 = jnp.zeros((cfg.M, cfg.M))
omega_num, _ = codesign.solve_modes(c0)
f_num = np.asarray(omega_num) / (2 * np.pi)

# Use actual thickness at c=0, not h0
h_actual = float(codesign.thickness(c0).mean())
D0 = cfg.E * h_actual**3 / (12 * (1 - cfg.nu**2))
rho_h_actual = cfg.rho * h_actual

analytical = []
for p in range(1, 5):
    for q in range(1, 5):
        w_sq = (D0 / rho_h_actual) * ((p * np.pi / cfg.Lx)**2 + (q * np.pi / cfg.Ly)**2)**2
        analytical.append((p, q, np.sqrt(w_sq) / (2 * np.pi)))
analytical.sort(key=lambda t: t[2])

print(f"{'(p,q)':>8}   {'analytic (Hz)':>15}   {'numeric (Hz)':>15}   {'err %':>8}")
for i, (p, q, f_ana) in enumerate(analytical[:cfg.n_modes]):
    err = 100 * (f_num[i] - f_ana) / f_ana
    print(f'  ({p},{q})   {f_ana:15.2f}   {f_num[i]:15.2f}   {err:7.2f}')

## 2. Passive-only baseline

Force control weights so high that optimal feedback gains vanish. The optimizer then only has the thickness field and actuator positions to work with, and the actuator positions do nothing useful since gains are ~0. This tells us how far shape alone can take us.

In [ ]:
import optax

def loss_passive(params, target, freqs):
    c, positions, _, _ = params
    log_q_fixed = jnp.log(1.0)
    log_r_fixed = jnp.log(1e10)
    return codesign.loss_fn((c, positions, log_q_fixed, log_r_fixed), target, freqs)

key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key)
c_init = 1e-4 * jax.random.normal(k1, (cfg.M, cfg.M))
pos_init = jnp.clip(0.5 + 0.15 * jax.random.normal(k2, (cfg.n_actuators, 2)), 0.1, 0.9)
params = (c_init, pos_init, jnp.log(1.0), jnp.log(1e10))

freqs, target = codesign.target_spectrum_example()

opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(1e-2))
state = opt.init(params)
vg_passive = jax.jit(jax.value_and_grad(loss_passive))

history_passive = []
best_loss_passive = float('inf')
params_passive = params

for step in range(300):
    loss, grads = vg_passive(params, target, freqs)
    
    # Track best loss and params
    if float(loss) < best_loss_passive:
        best_loss_passive = float(loss)
        params_passive = params
    
    updates, state = opt.update(grads, state, params)
    params = optax.apply_updates(params, updates)
    history_passive.append(float(loss))
    if step % 50 == 0:
        print(f'step {step:4d}   loss={float(loss):.4e}')

print(f'Best passive loss: {best_loss_passive:.4e}')

In [ ]:
# Compute the passive FRF for comparison later.
def compute_frf(params, freqs):
    c, positions, log_q, log_r = params
    omega, Phi = codesign.solve_modes(c)
    B = codesign.modal_input_matrix(Phi, positions)
    b_dist = codesign.modal_input_matrix(Phi, jnp.array([cfg.disturb_xy]))[:, 0]
    K_gain = codesign.modal_lqr_gains(omega, B, jnp.exp(log_q), jnp.exp(log_r))
    A_cl, B_d, C_out = codesign.closed_loop(omega, B, K_gain, b_dist)
    return codesign.frf_magnitude(A_cl, B_d, C_out, freqs)

omega_passive, _ = codesign.solve_modes(params_passive[0])
omega_hz_passive = (np.asarray(omega_passive) / (2 * np.pi)).tolist()
H_passive = compute_frf(params_passive, freqs)

## 3. Full co-design

All four design blocks are now free: thickness coefficients, actuator positions, $\log q$, and $\log r$.

In [ ]:
params_codesigned, best_loss_codesigned, history_codesigned, freqs, target = codesign.run(num_steps=400, seed=0)

c, positions, log_q, log_r = params_codesigned
omega_codesigned, _ = codesign.solve_modes(c)
omega_hz_codesigned = (np.asarray(omega_codesigned) / (2 * np.pi)).tolist()

H_codesigned = compute_frf(params_codesigned, freqs)
print(f'Best co-design loss: {best_loss_codesigned:.4e}')

## 4. Hero figure: passive vs co-designed

In [ ]:
fig = plots.dashboard(params_codesigned, history_codesigned, freqs, target,
                      baseline_frf=H_passive, optimized_frf=H_codesigned)
fig.savefig('hero.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Mode shapes with final actuator layout
fig = plots.plot_mode_shapes(params_codesigned[0], positions=params_codesigned[1], n_show=6)
fig.savefig('modes.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Quantify the co-design delta

The key finding for the report: how much did adding control actually help beyond pure shape optimization?

In [ ]:
improvement = 100 * (best_loss_passive - best_loss_codesigned) / best_loss_passive
print(f'Passive-only best loss:  {best_loss_passive:.4e}')
print(f'Co-designed best loss:   {best_loss_codesigned:.4e}')
print(f'Improvement from control:  {improvement:.1f} %')

In [ ]:
import json
import numpy as np

results = {
    "passive_best_loss": float(best_loss_passive),
    "codesign_best_loss": float(best_loss_codesigned),
    "improvement_pct": float(improvement),
    "passive_history": [float(x) for x in history_passive],
    "codesign_history": [float(x) for x in history_codesigned],
    "omega_hz_codesigned": omega_hz_codesigned,   # already numpy, captured earlier
    "omega_hz_passive": omega_hz_passive,
    "actuator_positions": np.asarray(params_codesigned[1]).tolist(),
    "log_q_final": float(params_codesigned[2]),
    "log_r_final": float(params_codesigned[3]),
}

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

np.savez("best_params.npz",
    c=np.asarray(params_codesigned[0]),
    positions=np.asarray(params_codesigned[1]),
    log_q=np.asarray(params_codesigned[2]),
    log_r=np.asarray(params_codesigned[3]),
    history_passive=np.array(history_passive),
    history_codesigned=np.array(history_codesigned),
    freqs=np.asarray(freqs),
    target=np.asarray(target),
    H_passive=np.asarray(H_passive),        # add these two
    H_codesigned=np.asarray(H_codesigned),  # add these two
)
print("Saved results.json and best_params.npz")